In [ ]:
from pathlib import Path
import importlib.util
import traceback
import json
import joblib
import pandas as pd
from flask import Flask, request, jsonify
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DEPLOY = ROOT / "artifacts" / "deployment"
MODEL_PATH = DEPLOY / "fake_profile_detector.pkl"
FEATURES_PATH = DEPLOY / "model_features.pkl"
THRESHOLDS_PATH = DEPLOY / "risk_thresholds.json"
EXPLANATION_PATH = DEPLOY / "explanation.py"

In [ ]:
# Load model with diagnostics; prefer matching kernel with scikit-learn 1.7.1
print("NOTE: If you installed scikit-learn 1.7.1, ensure the notebook kernel uses that Python environment before re-running this cell.")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

print('Model path exists, size (bytes):', MODEL_PATH.stat().st_size)
try:
    import sklearn
    print('sklearn version (runtime):', sklearn.__version__)
except Exception:
    print('Could not import sklearn to show version')

try:
    model = joblib.load(MODEL_PATH)
    print('Model loaded successfully:', type(model))
except Exception as e:
    print('Failed to load model with joblib:', repr(e))
    traceback.print_exc()
    # Show a small sample of the file header to help debugging
    try:
        with open(MODEL_PATH, 'rb') as fh:
            head = fh.read(256)
        print('File header (utf-8 safe):')
        print(head.decode('utf-8', errors='replace'))
    except Exception as e2:
        print('Could not read file header:', e2)
    print('\nHints:')
    print('- Kernel may not match the Python environment where scikit-learn 1.7.1 is installed.')
    print('- Switch kernel to the Python interpreter that has scikit-learn 1.7.1, then re-run this cell.')
    print('- Or re-generate the model artifact using the current environment or convert it to a portable format (ONNX).')
    print('Continuing without raising; `model` may be undefined if load failed.')

NOTE: If you installed scikit-learn 1.7.1, ensure the notebook kernel uses that Python environment before re-running this cell.
Model path exists, size (bytes): 3093481
sklearn version (runtime): 1.7.1
Model loaded successfully: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [ ]:
# Load features, thresholds, explanation module, define helpers and Flask app
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Features file not found at {FEATURES_PATH}")
features = joblib.load(FEATURES_PATH)

if not THRESHOLDS_PATH.exists():
    raise FileNotFoundError(f"Thresholds file not found at {THRESHOLDS_PATH}")
with open(THRESHOLDS_PATH, "r") as fh:
    thresholds = json.load(fh)

if not EXPLANATION_PATH.exists():
    explanation = None
else:
    spec = importlib.util.spec_from_file_location(
            "explaination",
            EXPLANATION_PATH
        )

    if spec is None or spec.loader is None:
        raise ImportError(
            f"Could not load explaination module from {EXPLANATION_PATH}"
        )
    explanation = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(explanation)

def _to_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default

def preprocess_input(payload):
    """Map user-friendly/raw payload keys to model feature names."""
    processed = {}
    # profile picture: accept 'has_profile_pic' (yes/no) or 'profile_pic' (0/1)
    pic = payload.get("has_profile_pic", payload.get("profile_pic", None))
    if isinstance(pic, str):
        processed["profile_pic"] = 0 if pic.strip().lower() in ("no", "n", "false", "0") else 1
    else:
        try:
            processed["profile_pic"] = int(pic) if pic is not None else 0
        except Exception:
            processed["profile_pic"] = 0

    # username digit ratio (capped by length to avoid divide-by-zero)
    username = str(payload.get("username", ""))
    processed["username_num_length_capped"] = sum(c.isdigit() for c in username) / max(len(username), 1)

    # description / bio length
    bio = str(payload.get("bio", payload.get("description", "")))
    processed["description_length_capped"] = len(bio)

    # followers / following ratio
    followers = _to_int(payload.get("followers", payload.get("follower_count", 0)), 0)
    following = _to_int(payload.get("following", payload.get("following_count", 1)), 1)
    processed["followers_following_ratio_clipped"] = followers / max(following, 1)

    # number of posts
    processed["num_posts_capped"] = _to_int(payload.get("posts", payload.get("num_posts", 0)), 0)

    # Preserve any already-provided model-feature keys (idempotent behavior)
    for k, v in payload.items():
        if k in features:
            processed[k] = v

    return processed

def prepare_row(payload):
    # First, map raw user-friendly keys into model feature names (idempotent)
    processed = preprocess_input(payload) if isinstance(payload, dict) else {}
    df = pd.DataFrame([processed])
    # Ensure columns are in the expected order and missing columns are filled with NaN
    for col in features:
        if col not in df.columns:
            df[col] = pd.NA
    # Reorder to model features
    df = df[features]
    # Coerce numeric where possible and fill NaNs with 0 for inference
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.fillna(0)
    return df

def score_row(df_row):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(df_row)[:, 1]
        prob = float(proba[0])
        score = int(round(prob * 100))
    else:
        pred = model.predict(df_row)[0]
        prob = float(pred)
        score = int(round(prob * 100))

    low_thr = int(thresholds.get("low", 30))
    med_thr = int(thresholds.get("medium", 70))

    if score >= med_thr:
        label = "High Risk"
    elif score >= low_thr:
        label = "Medium Risk"
    else:
        label = "Low Risk"

    return score, label, prob


def calculate_input_quality(payload: dict) -> int:
    important_fields = [
        "username",
        "fullname",
        "bio",
        "followers",
        "following",
        "posts",
        "has_profile_pic",
        "private",
    ]
    present = 0
    for f in important_fields:
        if f not in payload:
            continue
        v = payload.get(f)
        if v is None:
            continue
        if isinstance(v, str) and str(v).strip() == "":
            continue
        present += 1
    score = round((present / len(important_fields)) * 100)
    return int(score)


def get_model_confidence(risk_score: int, input_quality: int, prob: float) -> str:
    if input_quality < 40:
        return "Low"
    if risk_score >= 85 and input_quality >= 70:
        return "High"
    if prob >= 0.75:
        return "High"
    if prob >= 0.6:
        return "Medium"
    return "Low"

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json()
    if payload is None:
        return jsonify({"error": "Invalid JSON payload"}), 400

    try:
        row = prepare_row(payload)
        score, label, prob = score_row(row)
    except Exception as e:
        return jsonify({"error": "Scoring failed", "detail": str(e)}), 500

    reasons = []
    try:
        if explanation and hasattr(explanation, 'generate_reasons'):
            reasons = explanation.generate_reasons(row.iloc[0].to_dict())
    except Exception:
        reasons = []

    input_quality_score = calculate_input_quality(payload if isinstance(payload, dict) else {})
    model_confidence = get_model_confidence(score, input_quality_score, prob if 'prob' in locals() else 0.0)
    response = {
        "risk_score": score,
        "status": label,
        "input_quality_score": input_quality_score,
        "model_confidence": model_confidence,
        "reasons": reasons
    }
    if input_quality_score < 40:
        response["warning"] = "Prediction based on limited profile information"

    return jsonify(response)


In [31]:
from werkzeug.serving import run_simple

run_simple('0.0.0.0', 5000, app)

 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.18.3.176:5000
Press CTRL+C to quit
